In [ ]:
!pip install -U vllm==0.22.0

In [ ]:
# 安装 transformers
!pip install -U transformers accelerate huggingface_hub

In [ ]:
!python -m pip show vllm

In [ ]:
!apt-get update
!apt-get install -y nvidia-cuda-dev

In [1]:
%%writefile vllm.sh
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --max-num-seqs 1 \
  --max-model-len 1024 \
  --attention-backend TRITON_ATTN \
  --gpu-memory-utilization 0.85 \
  --tensor-parallel-size 1 &

Writing vllm.sh


In [ ]:
# 查看服务是否启动
!netstat -ant

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",  # vLLM 默认不校验
    base_url="http://127.0.0.1:8000/v1",
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",  # 必须与启动服务时加载的模型名称一致
    messages=[
        {"role": "user", "content": "AI软件基础包括哪些核心内容？"}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI

# vLLM 服务地址
client = OpenAI(
    api_key="EMPTY",  # vLLM 默认不校验
    base_url="http://127.0.0.1:8000/v1",
)

# 对话历史
messages = [
    {
        "role": "system",
        "content": "你是一个有帮助的AI助手"
    }
]


def chat_with_qwen(user_input: str):
    # 加入用户消息
    messages.append({
        "role": "user",
        "content": user_input
    })

    # 调用 vLLM
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=messages,
        temperature=0.7,
        max_tokens=512,
    )

    # 取回复
    reply = response.choices[0].message.content

    # 加入历史
    messages.append({
        "role": "assistant",
        "content": reply
    })

    return reply


if __name__ == "__main__":
    print("=== Local Qwen (vLLM) ===")

    while True:
        user_input = input("You: ")

        if user_input.strip().lower() in {"exit", "quit"}:
            break

        reply = chat_with_qwen(user_input)

        print("Qwen:", reply)
